```text
What this script does per new essay:
1) Retrieve top-2 similar marked essays from bank (TF-IDF cosine similarity).
2) Ask LLM to compare (better/worse/similar) and estimate mark deltas vs each neighbour.
3) Convert deltas into a numeric prior.
4) Ask LLM to grade with rubric + neighbours + prior, output strict JSON.
```

In [ ]:
#“RAG plus pairwise calibration”, and it addresses two common problems in pure RAG grading: scale
#drift and over-reliance on exemplars.

import os, re, json, glob, math
from collections import Counter, defaultdict

import torch
from docx import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

In [ ]:
ESSAY_DIR = "/path/to/project/essays"
EXAMPLES_DIR = os.path.join(ESSAY_DIR, "examples")

# Pick one (same pool you use already)
#MODEL_DIR = "/path/to/project/models/qwen2.5-7b"
# MODEL_DIR = "/path/to/project/models/deepseek-7b"
MODEL_DIR = "/path/to/project/models/deepseek-qwen14b-4bit"
# MODEL_DIR  = "/path/to/project/models/phi-4"

OUTPUT_DIR = "/path/to/project/grader5_outputs"


MARKING_SCHEME = r"""Marking rubric (weights sum to 100):
- Overall presentation and quality of communication 20%, 
- Evidence of continuous reflection during your study 20% ,
- Breadth of study, including degree of engagement with others, including students and university staff 30%,
- Linking your studies with project management practices (both your own and more generally across the profession) 30%.
When marking consider the following ranges:
0–49 = Fail (criteria not met or met at an inadequate level)
50–59 = Pass (criteria met at a basic level)
60–69 = Merit (criteria met well, with good understanding and analysis)
70+ = Distinction (criteria met at an excellent level, showing originality, critical insight, and clear argument)
Return ONLY JSON."""

In [ ]:
# -------------------------
# Small helpers
# -------------------------

def read_docx(path: str) -> str:
    d = Document(path)
    return "\n".join(p.text.strip() for p in d.paragraphs if p.text and p.text.strip()).strip()

def mkdirp(p: str) -> None:
    os.makedirs(p, exist_ok=True)

def extract_json(text: str):
    t = text.strip()
    if t.startswith("{") and t.endswith("}"):
        try: return json.loads(t)
        except Exception: pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

def clip_int(x, lo, hi, default=0):
    try: v = int(x)
    except Exception: v = default
    return max(lo, min(hi, v))

def grade_band(mark: int) -> str:
    if mark >= 70: return "Distinction"
    if mark >= 60: return "Merit"
    if mark >= 50: return "Pass"
    return "Fail"

def excerpt(s: str, n: int = 1400) -> str:
    s = s.strip()
    return s if len(s) <= n else (s[:n].rstrip() + "...")

_word = re.compile(r"[A-Za-z0-9']+")

def tokens(text: str):
    return [w.lower() for w in _word.findall(text) if len(w) >= 2]

In [ ]:
# -------------------------
# Examples loader (like GRADER2)
# -------------------------

def extract_mark_and_feedback(fb_path: str):
    text = read_docx(fb_path)
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    mark = None
    fb = []
    capture = False

    for line in lines:
        if line.startswith("MARK"):
            m = re.search(r"MARK\s*[:\-]?\s*(\d+)", line)
            if m: mark = int(m.group(1))
        elif line.startswith("FEEDBACK"):
            capture = True
        elif capture:
            fb.append(line)

    return mark, " ".join(fb).strip()

def load_examples(examples_dir: str):
    essay_files = [
        f for f in sorted(os.listdir(examples_dir))
        if f.endswith(".docx") and not f.endswith("_FB.docx")
    ]

    ex = []
    for f in essay_files:
        eid = os.path.splitext(f)[0]
        essay_path = os.path.join(examples_dir, f)
        fb_path = os.path.join(examples_dir, f"{eid}_FB.docx")
        if not os.path.exists(fb_path):
            continue

        etxt = read_docx(essay_path)
        mark, fb = extract_mark_and_feedback(fb_path)
        if not etxt or mark is None:
            continue

        ex.append({"id": eid, "text": etxt, "mark": clip_int(mark, 0, 100), "feedback": fb})

    return ex

In [ ]:
# -------------------------
# Tiny TF-IDF retriever
# TF-IDF + cosine similarity retriever
# whose only job is to answer:
# “which two past essays look most like this one?”
# -------------------------

def build_tfidf(examples):

#Takes all marked example essays.
#Tokenises each essay into words.
#Counts how often each word appears in each essay (term frequency).
#Counts in how many essays each word appears overall (document frequency).
#Computes an IDF weight so common words matter less and distinctive words matter more.
#Builds one sparse vector per essay (TF × IDF) and stores its length (norm).
#Returns everything needed to compare new essays later.

    N = len(examples)
    dfs = defaultdict(int)
    doc_counts = []
    doc_norm = []
    doc_vec = []

    for e in examples:
        c = Counter(tokens(e["text"]))
        doc_counts.append(c)
        for term in c.keys():
            dfs[term] += 1

    idf = {t: (math.log((N + 1) / (df + 1)) + 1.0) for t, df in dfs.items()}

    for c in doc_counts:
        v = {}
        for t, tf in c.items():
            w = tf * idf.get(t, 0.0)
            if w:
                v[t] = w
        nrm = math.sqrt(sum(w*w for w in v.values())) or 1.0
        doc_vec.append(v)
        doc_norm.append(nrm)

    return idf, doc_vec, doc_norm



def top2_sim(query_text: str, idf, doc_vec, doc_norm, examples):
#Tokenises the new essay.
#Builds its TF-IDF vector using the same IDF weights.
#Computes cosine similarity between the new essay vector and each example essay vector.
#Sorts by similarity and returns the top two examples.
    
    qc = Counter(tokens(query_text))
    qv = {}
    for t, tf in qc.items():
        w = tf * idf.get(t, 0.0)
        if w:
            qv[t] = w
    qn = math.sqrt(sum(w*w for w in qv.values())) or 1.0

    scored = []
    for i, dv in enumerate(doc_vec):
        dot = 0.0
        for t, w in qv.items():
            dot += w * dv.get(t, 0.0)
        sim = dot / (qn * doc_norm[i])
        scored.append((sim, examples[i]))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:2]

In [ ]:
# -------------------------
# Local model
# -------------------------

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tok = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True, trust_remote_code=True)
mdl = AutoModelForCausalLM.from_pretrained(MODEL_DIR, quantization_config=bnb, device_map="auto", trust_remote_code=True)

if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id

gen = pipeline("text-generation", model=mdl, tokenizer=tok, device_map="auto", torch_dtype=torch.float16)

def chat(system_text: str, user_text: str) -> str:
    if hasattr(tok, "apply_chat_template") and getattr(tok, "chat_template", None):
        try:
            return tok.apply_chat_template(
                [{"role": "system", "content": system_text}, {"role": "user", "content": user_text}],
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            pass
    return system_text + "\n\n" + user_text

def llm(system_text: str, user_text: str, max_new_tokens: int) -> str:
    out = gen(
        chat(system_text, user_text),
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.4,
        top_p=0.95,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
        return_full_text=False,
    )
    return out[0]["generated_text"].strip()


In [ ]:
# -------------------------
# Prompts
# -------------------------

SYSTEM = "You are a careful university marker. Return ONLY JSON."

def prompt_compare(new_txt: str, A, Asim: float, B, Bsim: float) -> str:
    return f"""
RUBRIC:
{MARKING_SCHEME}

Compare the NEW essay to two marked examples and estimate mark deltas.

Rules:
- Output ONLY JSON.
- delta_A and delta_B are integers in [-15, +15].
- Evidence must quote ONLY the NEW essay.

EXAMPLE A (mark={A['mark']}, sim={Asim:.3f}):
{excerpt(A['text'])}

EXAMPLE B (mark={B['mark']}, sim={Bsim:.3f}):
{excerpt(B['text'])}

NEW ESSAY:
{new_txt}

Return JSON:
{{
  "relative_to_A":"better|worse|similar",
  "delta_A":0,
  "relative_to_B":"better|worse|similar",
  "delta_B":0,
  "confidence":"low|medium|high",
  "evidence":["...", "..."]
}}
""".strip()

def prior_from(A, Asim: float, dA: int, B, Bsim: float, dB: int) -> float:
    wA, wB = max(0.0, Asim), max(0.0, Bsim)
    a = float(A["mark"] + dA)
    b = float(B["mark"] + dB)
    if wA + wB > 0:
        p = (wA * a + wB * b) / (wA + wB)
    else:
        p = 0.5 * (a + b)
    return max(0.0, min(100.0, p))

def prompt_grade(new_txt: str, A, Asim: float, B, Bsim: float, comp: dict, prior: float) -> str:
    return f"""
RUBRIC:
{MARKING_SCHEME}

FEW-SHOT CALIBRATION (do not copy wording, use to calibrate standard):

EXAMPLE A (mark={A['mark']} sim={Asim:.3f})
ESSAY:
{excerpt(A['text'], 1200)}
FEEDBACK:
{A['feedback']}

EXAMPLE B (mark={B['mark']} sim={Bsim:.3f})
ESSAY:
{excerpt(B['text'], 1200)}
FEEDBACK:
{B['feedback']}

PAIRWISE CALIBRATION (your earlier comparison, JSON):
{json.dumps(comp, ensure_ascii=False)}

PRIOR MARK (computed from calibration):
{prior:.1f}

TASK:
Mark the NEW essay using the rubric.
Use the examples as calibration shots (essay + mark + feedback).
Also keep the prior in mind: stay near it unless rubric evidence supports moving away.
If you move by more than 5 marks, explain why.
Evidence must quote ONLY the NEW essay.

NEW ESSAY:
{new_txt}

Return ONLY JSON:
{{
  "mark": 0,
  "grade_band": "Fail|Pass|Merit|Distinction",
  "feedback": "...",
  "evidence": ["...", "...", "..."],
  "moved_from_prior_by": 0,
  "reason_if_moved": ""
}}
""".strip()

#Caveat: If your essays are long extract only the most informative paragraphs.
#Otherwise, you will hit context limits, especially with smaller 7B models.


In [ ]:
# -------------------------
# Run
# -------------------------

mkdirp(OUTPUT_DIR)

examples = load_examples(EXAMPLES_DIR)
if len(examples) < 2:
    raise RuntimeError(f"Need at least 2 valid examples in {EXAMPLES_DIR} (CODE.docx + CODE_FB.docx).")

idf, doc_vec, doc_norm = build_tfidf(examples)

essay_paths = sorted(glob.glob(os.path.join(ESSAY_DIR, "*.docx")))
essay_paths = [p for p in essay_paths if not p.endswith("_FB.docx")]  # ignore any feedback docs in root

MAX_RETRIES = 6

for p in essay_paths:
    eid = os.path.splitext(os.path.basename(p))[0]
    new_txt = read_docx(p)
    if not new_txt:
        print(f"[SKIP] {eid} (empty)")
        continue

    out = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            top2 = top2_sim(new_txt, idf, doc_vec, doc_norm, examples)
            if len(top2) < 2:
                print(f"[SKIP] {eid} (no neighbours)")
                break

            (Asim, A), (Bsim, B) = top2[0], top2[1]

            # --- Step 1: pairwise comparison ---
            comp_raw = llm(SYSTEM, prompt_compare(new_txt, A, Asim, B, Bsim), max_new_tokens=512)
            comp = extract_json(comp_raw)
            if comp is None:
                raise ValueError("compare step returned no JSON")

            comp_clean = {
                "relative_to_A": str(comp.get("relative_to_A", "similar")).lower() if str(comp.get("relative_to_A", "similar")).lower() in {"better","worse","similar"} else "similar",
                "delta_A": clip_int(comp.get("delta_A", 0), -15, 15, 0),
                "relative_to_B": str(comp.get("relative_to_B", "similar")).lower() if str(comp.get("relative_to_B", "similar")).lower() in {"better","worse","similar"} else "similar",
                "delta_B": clip_int(comp.get("delta_B", 0), -15, 15, 0),
                "confidence": str(comp.get("confidence", "medium")).lower() if str(comp.get("confidence", "medium")).lower() in {"low","medium","high"} else "medium",
                "evidence": comp.get("evidence", []),
            }

            prior = prior_from(A, Asim, comp_clean["delta_A"], B, Bsim, comp_clean["delta_B"])

            # --- Step 2: final grading ---
            final_raw = llm(SYSTEM, prompt_grade(new_txt, A, Asim, B, Bsim, comp_clean, prior), max_new_tokens=1024)
            final = extract_json(final_raw)
            if final is None:
                raise ValueError("final grade step returned no JSON")

            mark = final.get("mark")
            mark = clip_int(mark, 0, 100, 0) if mark is not None else None

            out = {
                "essay_id": eid,
                "model_dir": MODEL_DIR,
                "retrieved_examples": [
                    {"essay_id": A["id"], "mark": A["mark"], "sim": Asim},
                    {"essay_id": B["id"], "mark": B["mark"], "sim": Bsim},
                ],
                "pairwise_calibration": comp_clean,
                "prior_mark": prior,
                "final": {
                    "mark": mark,
                    "grade_band": final.get("grade_band") or (grade_band(mark) if mark is not None else None),
                    "feedback": str(final.get("feedback", "")),
                    "evidence": final.get("evidence", []),
                    "moved_from_prior_by": final.get("moved_from_prior_by"),
                    "reason_if_moved": str(final.get("reason_if_moved", "")),
                },
            }
            break  # success, exit retry loop

        except Exception as e:
            print(f"Attempt {attempt}/{MAX_RETRIES} for {eid} failed: {e}")

    # --- After retry loop ---
    if out is None:
        print(f"{eid} SKIPPED after {MAX_RETRIES} attempts.\n")
        continue

    # --- Save and print ---
    with open(os.path.join(OUTPUT_DIR, f"{eid}.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"--- BEGIN {eid}.docx ---")
    print(
        f"EXAMPLES USED: "
        f"{A['id']} (mark={A['mark']}, sim={Asim:.3f}), "
        f"{B['id']} (mark={B['mark']}, sim={Bsim:.3f})"
    )
    print(f"FINAL GRADE: {mark}")
    print("\nFEEDBACK:\n" + str(final.get("feedback", "")).strip())
    print(f"--- END {eid}.docx ---\n")
